# PDF Deciphering Assistant using RAG
### Course 4 Capstone Project
### Building an AI-powered scientific PDF understanding system

## Project Setup and PDF Collection

The PDF Deciphering Assistant begins by loading scientific PDF papers from the local data directory. These papers form the knowledge base used for semantic retrieval and question answering.

In [2]:
import sys
import os

print("Hello from PDF Deciphering Assistant!")
print("Python version:", sys.version)
print("Current working directory:", os.getcwd())

Hello from PDF Deciphering Assistant!
Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Current working directory: c:\Users\Monic\projects\pdf_deciphering_assistant\notebooks


## Import Required Libraries
Load all Python packages required for PDF loading, chunking, embeddings, vector storage, and retrieval.

In [3]:
import os

pdf_folder = "../data"

pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]

print("PDF files found:")
for pdf in pdf_files:
    print("-", pdf)

print("\nTotal PDFs found:", len(pdf_files))

PDF files found:
- paper1.pdf
- paper2.pdf
- paper3.pdf
- paper4.pdf
- paper5.pdf

Total PDFs found: 5


## Load All PDF Documents

Find all PDF files in the data folder, load every page with PyPDFLoader, and combine them into one document collection.

In [5]:
from langchain_community.document_loaders import PyPDFLoader
import os

pdf_folder = "../data"

# Use all five PDFs
pdf_files = sorted(
    file for file in os.listdir(pdf_folder)
    if file.lower().endswith(".pdf")
)

documents = []

for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    loader = PyPDFLoader(pdf_path)
    pdf_pages = loader.load()
    documents.extend(pdf_pages)

    print(f"Loaded {pdf_file}: {len(pdf_pages)} pages")

print("\nTotal PDFs loaded:", len(pdf_files))
print("Total pages loaded:", len(documents))

C:\Users\Monic\AppData\Local\Temp\ipykernel_17352\583485704.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded paper1.pdf: 23 pages
Loaded paper2.pdf: 3 pages
Loaded paper3.pdf: 8 pages
Loaded paper4.pdf: 45 pages
Loaded paper5.pdf: 18 pages

Total PDFs loaded: 5
Total pages loaded: 97


## Split PDF Pages into Overlapping Text Chunks

Use `RecursiveCharacterTextSplitter` to divide the loaded PDF pages into smaller overlapping chunks for embedding and semantic retrieval. This step also calculates the total number of chunks, the average chunk size, and displays a preview of the first chunk.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

chunk_sizes = [len(chunk.page_content) for chunk in chunks]
average_chunk_size = sum(chunk_sizes) / len(chunk_sizes)

print("Original pages:", len(documents))
print("Total chunks created:", len(chunks))
print(f"Average chunk size: {average_chunk_size:.2f} characters")

print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:500])

Original pages: 97
Total chunks created: 399
Average chunk size: 887.30 characters

First chunk preview:

An Introduction
to Restricted Boltzmann Machines
Asja Fischer1,2 and Christian Igel2
1 Institut f¨ur Neuroinformatik, Ruhr-Universit¨at Bochum, Germany
2 Department of Computer Science, University of Copenhagen, Denmark
Abstract. Restricted Boltzmann machines (RBMs) are probabilistic
graphical models that can be interpreted as stochastic neural networks.
Theincrease incomputational powerand thedevelopmentoffaster learn-
ing algorithms have made them applicable to relevant machine learning
proble


## Load OpenAI API Key
Load the OpenAI API key from the .env file for embeddings and GPT access.

In [7]:
from dotenv import load_dotenv
import os

# Our notebook is inside notebooks/, so .env is one level up
load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OpenAI API key loaded successfully.")
    print("Key starts with:", api_key[:7])
else:
    print("OpenAI API key NOT found. Please check your .env file.")

OpenAI API key loaded successfully.
Key starts with: sk-proj


## Create OpenAI Embeddings for Semantic Search

Initialize the OpenAI embedding model (`text-embedding-3-small`) and convert sample text into a numerical embedding vector. This verifies that the embedding model is working correctly before building the FAISS vector database.

In [10]:
from langchain_openai import OpenAIEmbeddings

# Create the embedding model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Test embedding creation
test_embedding = embeddings.embed_query(
    "What is a Restricted Boltzmann Machine?"
)

print("Embedding created successfully!")
print("Embedding vector length:", len(test_embedding))
print("First 10 numbers:")
print(test_embedding[:10])

Embedding created successfully!
Embedding vector length: 1536
First 10 numbers:
[0.0218353271484375, 0.0161590576171875, -0.0013875961303710938, -0.0189666748046875, -0.03594970703125, -0.00020515918731689453, -0.00974273681640625, 0.000720977783203125, -0.054290771484375, 0.017181396484375]


## Build the FAISS Vector Store with OpenAI Embeddings

Create a FAISS vector database using OpenAI's `text-embedding-3-small` embedding model for semantic search over PDF chunks. Unlike the later local pipeline, this approach relies on OpenAI embeddings rather than SBERT (`all-MiniLM-L6-v2`) embeddings.

In [11]:
from langchain_community.vectorstores import FAISS

# Create FAISS vector database from your chunks
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS vectorstore created successfully!")
print("Total chunks stored:", len(chunks))

FAISS vectorstore created successfully!
Total chunks stored: 399


## Retrieve the Best PDF Passage and Generate an RBM Explanation

This pipeline demonstrates Retrieval-Augmented Generation (RAG) using OpenAI embeddings and FAISS. First, PDF chunks are converted into OpenAI `text-embedding-3-small` vectors and stored in a FAISS vector database. When the user asks *"What is a Restricted Boltzmann Machine?"*, FAISS performs semantic similarity search and retrieves the most relevant chunk. GPT-4.1-mini then receives only this retrieved passage as context and generates an explanation based on the document rather than relying on its general knowledge.

This earlier implementation uses OpenAI embeddings instead of the later SBERT (`all-MiniLM-L6-v2`) local embedding model.

In [12]:
from openai import OpenAI

query = "What is a Restricted Boltzmann Machine?"

# Retrieve the best PDF passage first
results = vectorstore.similarity_search(
    query,
    k=1
)

best_result = results[0]
context = best_result.page_content

client = OpenAI()

prompt = f"""
You are a patient technical teacher.

Answer the question using only the retrieved PDF passage.

Question:
{query}

Retrieved PDF passage:
{context}

Instructions:
1. Explain it in plain English.
2. Give a short technical definition.
3. Explain unfamiliar terminology.
4. If the passage is insufficient, say so clearly.
"""

response = client.responses.create(
    model="gpt-5-mini",
    input=prompt
)

print("QUESTION:")
print(query)

print("\nSOURCE:")
print(best_result.metadata)

print("\nDECIPHERED ANSWER:")
print(response.output_text)

QUESTION:
What is a Restricted Boltzmann Machine?

SOURCE:
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'LaTeX with hyperref package', 'creationdate': '2012-08-02T16:35:12+05:30', 'author': '', 'keywords': '', 'moddate': '2026-07-09T12:24:40+06:00', 'subject': '', 'title': '', 'source': '../data\\paper1.pdf', 'total_pages': 23, 'page': 11, 'page_label': '25'}

DECIPHERED ANSWER:
Plain-English explanation
- A Restricted Boltzmann Machine (RBM), as shown in this passage, is a simple two-layer probabilistic model with a layer of visible units (the variables you observe) and a layer of hidden units (latent variables). Each visible unit interacts only with the hidden units (and vice versa), not with other units in the same layer. Because of that structure, once you fix the states of one layer (say the hidden units), every unit in the other layer becomes independent of the others and can be sampled at once. The passage shows how the probability that a particular visible unit

## Simplify an OpenAI-FAISS Retrieved PDF Passage with GPT-4.1-mini

Use the best PDF passage previously retrieved through OpenAI `text-embedding-3-small` embeddings and FAISS semantic similarity search. The retrieved passage is supplied to GPT-4.1-mini as context, and GPT-4.1-mini rewrites the difficult scientific text in simple language.

In this section, semantic retrieval uses OpenAI embeddings rather than the later SBERT (`all-MiniLM-L6-v2`) embedding pipeline.

### Retrieval and Explanation Flow

```text
PDF chunks
   ↓
OpenAI text-embedding-3-small embeddings
   ↓
FAISS vector database
   ↓
Semantic similarity search
   ↓
Most relevant passage stored in results[0]
   ↓
GPT-4.1-mini receives retrieved context
   ↓
Plain-English scientific explanation
```

In [14]:
from openai import OpenAI

client = OpenAI()

context = results[0].page_content

prompt = f"""
Using the following text from a scientific paper, explain it in simple language.

Text:
{context}

Explanation:
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

answer = response.choices[0].message.content

print(answer)

Sure! Here's a simple explanation of the text about Restricted Boltzmann Machines (RBMs):

---

The text is discussing how to calculate the probability that one particular "visible unit" (think of it as one part of the data we observe) is turned on (value = 1), given the current state of the "hidden units" (which capture patterns or features) in an RBM.

1. **Breaking down the problem:**  
   - Suppose you have many visible units, but we want to focus on just one, called the **lth visible unit**.  
   - They define two quantities:  
     - **αl(h):** This sums up how strongly each hidden unit is connected to this particular visible unit (plus a bias term).  
     - **β(v−l,h):** This sums up all the other connections and biases that involve the other visible units (not the lth one) and hidden units.

2. **Energy function:**  
   RBMs use an **energy function** to describe how compatible a given configuration of visible and hidden units is (lower energy means more likely). The energy fo

## Verify Active Python Environment

Confirm that the notebook is running inside the correct `rag_capstone` Python environment.

In [15]:
import sys

print("Python executable:")
print(sys.executable)

Python executable:
d:\ANACONDA\envs\rag_capstone\python.exe


## Verify Current Working Directory

Check the current notebook working directory to ensure file paths are resolved correctly.

In [16]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
c:\Users\Monic\projects\pdf_deciphering_assistant\notebooks


## Verify PDF Files in Data Folder

List all files inside the `data` directory to confirm that the PDF documents are accessible.

In [17]:
import os

print(os.listdir("../data"))

['paper1.pdf', 'paper2.pdf', 'paper3.pdf', 'paper4.pdf', 'paper5.pdf']


## Display Available PDF Documents

Print the names of all PDF files that will be processed by the PDF Deciphering Assistant.

In [18]:
import os

print("PDF files found:")

for file in os.listdir("../data"):
    print(file)

PDF files found:
paper1.pdf
paper2.pdf
paper3.pdf
paper4.pdf
paper5.pdf


## Install Local Embedding Dependencies for SBERT-Based Retrieval

Install the `sentence-transformers` library and the LangChain Hugging Face integration package required for local embedding generation.

These packages enable the use of SBERT (Sentence-BERT) models such as `all-MiniLM-L6-v2`, allowing the PDF Deciphering Assistant to generate embeddings locally instead of relying on OpenAI embedding APIs.

### Dependency Flow

```text
sentence-transformers
        +
langchain-huggingface
        ↓
SBERT model loading
(all-MiniLM-L6-v2)
        ↓
Local embedding generation
(384-dimensional vectors)
        ↓
FAISS vector database
        ↓
Semantic retrieval
        ↓
GPT-4.1-mini receives retrieved context
```

In [5]:
%pip install -U sentence-transformers langchain-huggingface

  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 596.4/596.4 kB 7.1 MB/s  0:00:00
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   ------- -------------------------------- 2.1/11.5 MB 10.7 MB/s eta 0:00:01
   --------------- ------------------------ 4.5/11.5 MB 10.8 MB/s eta 0:00:01
   ----------------------- ---------------- 6.8/11.5 MB 11.0 MB/s eta 0:00:01
   ------------------------------- -------- 9.2/11.5 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.5 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 11.5/11.5 MB 11.1 MB/s  0:00:01
   ---------------------------------------- 0.0/770.3 kB ? eta -:--:--
   ---------------------------------------- 770.3/770.3 kB 8.1 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----------------------- ---------

## Install Local Embedding Dependencies

Install the Sentence-BERT (SBERT) libraries and LangChain Hugging Face integration required for local semantic embeddings using the `all-MiniLM-L6-v2` model.

In [19]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Sentence Transformer loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Transformer loaded successfully!


## Install Jupyter Widget Dependencies

Install `ipywidgets` to enable interactive notebook widgets and improve compatibility with machine learning libraries.

In [2]:
%pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 10.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 11.2 MB/s  0:00:00

   ---------------------------------------- 0/3 [widgetsnbextension]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


## Create Local Embeddings and Build FAISS Vector Store

Convert PDF chunks into local embeddings using Sentence Transformers and store them in a FAISS vector database for semantic search.

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Connect LangChain to the local Sentence Transformer model
local_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Convert all PDF chunks into 384-dimensional embeddings
# and store them in a new FAISS vector database
local_vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=local_embeddings
)

print("Local Sentence Transformer FAISS database created successfully!")
print("Total chunks stored:", len(chunks))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Local Sentence Transformer FAISS database created successfully!
Total chunks stored: 399


## Retrieve Relevant PDF Passages Using Semantic Search

Search the local FAISS vector database and retrieve the most relevant document chunks for the user's question.

In [23]:
query = "What is Gibbs Sampling?"

docs = local_vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(docs, start=1):
    print("="*80)
    print(f"Result {i}")
    print("="*80)
    print(doc.page_content[:1500])
    print("\n")

Result 1
and aperiodic Markov chain with stationary distributionπ = q. This is a non-
trivial task. Ift is large enough, a sample X
(t) from the constructed chain is
then approximately a sample fromπ and therefore fromq. Gibbs Sampling [13]
is such a MCMC method and will be described in the following section.
3.2 Gibbs Sampling
Gibbs Sampling belongs to the class of Metropolis-Hastings algorithms [14]. It is
a simple MCMC algorithm for producing samples from the joint probability dis-
tribution of multiple random variables. The basic idea is to update each variable
subsequently based on its conditional distribution given the state of the others.
We will describe it in detail by explaining how Gibbs sampling can be used to
simulate the Gibbs distribution of an MRF.
We consider an MRFX =( X
1,...,X N) w.r.t. a graphG =( V,E ), where
V = {1,...,N } for the sake of clearness of notation. The random variablesXi,
i ∈ V take values in a ﬁnite setΛ and π(x)= 1
Ze−E(x) is the joint probability


## Perform Retrieval-Augmented Generation (RAG) with SBERT, FAISS, and GPT-4.1-mini

Provide GPT-4.1-mini with the PDF passages retrieved using SBERT (`all-MiniLM-L6-v2`) embeddings and FAISS semantic search so it can answer the user's question using information from the documents rather than general knowledge.

In this pipeline, SBERT converts both the user's question and the PDF chunks into 384-dimensional semantic embedding vectors. FAISS then performs efficient similarity search to retrieve the most relevant passages. These retrieved passages are supplied to GPT-4.1-mini as context, allowing the model to generate an answer grounded in the contents of the documents rather than relying on its pretrained knowledge alone.

### Retrieval and Generation Flow

```text
PDF chunks
   ↓
SBERT embeddings
(all-MiniLM-L6-v2)
   ↓
384-dimensional semantic vectors
   ↓
FAISS vector database
   ↓
Semantic similarity search
   ↓
Top 3 most relevant passages retrieved
(docs[0], docs[1], docs[2])
   ↓
Retrieved passages combined into context
(context variable)
   ↓
GPT-4.1-mini receives retrieved context
   ↓
Document-grounded answer generation
```

In [24]:
from openai import OpenAI

# Create the OpenAI client.
# It will use OPENAI_API_KEY from your .env/environment.
client = OpenAI()

# Join the three retrieved chunks into one context block.
context = "\n\n".join(
    f"Retrieved passage {i}:\n{doc.page_content}"
    for i, doc in enumerate(docs, start=1)
)

prompt = f"""
You are a patient technical teacher.

Answer the user's question using only the retrieved PDF passages below.

User question:
{query}

Retrieved PDF passages:
{context}

Instructions:
1. Explain the answer in plain English.
2. Give a short formal definition.
3. Explain the main steps of Gibbs sampling.
4. Explain unfamiliar terms such as MCMC and conditional distribution.
5. Do not invent information that is absent from the passages.
6. If the passages are insufficient, say so clearly.
"""

response = client.responses.create(
    model="gpt-5-mini",
    input=prompt
)

answer = response.output_text

print("QUESTION:")
print(query)

print("\nDECIPHERED ANSWER:")
print(answer)

QUESTION:
What is Gibbs Sampling?

DECIPHERED ANSWER:
Plain English answer
- Gibbs Sampling is a simple way to draw samples from a complicated joint probability distribution of many variables. You repeatedly update one variable at a time (or whole groups of variables) by drawing a new value from that variable’s probability distribution conditioned on the current values of all the other variables. After doing this many times, the values produced are approximately samples from the joint distribution you care about.

Short formal definition
- Gibbs Sampling is a Markov chain Monte Carlo (MCMC) algorithm (a member of the Metropolis–Hastings class) that produces samples from a joint distribution by repeatedly sampling each variable from its conditional distribution given the current states of the other variables.

Main steps of Gibbs sampling (as given in the passages)
1. Represent the joint random vector X = (X1,...,XN) (for example an MRF) whose joint distribution π(x) you want to sample 